# **TFM DSMarket**

In [1]:
#nota Sandra: usar entorno virtual en local venv_tf(python 3.12.10)

# pruebas github

In [2]:
import pandas as pd
from datetime import timedelta
import numpy as np
import plotly.express as px

import warnings
warnings.filterwarnings("ignore")

import holidays
import gc

In [3]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [52]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  DATA_PATH = '/content/drive/MyDrive/data_dsmarket/'
except ImportError:
    DATA_PATH = 'data_dsmarket/'


df_daily = pd.read_csv(DATA_PATH + 'daily_calendar_with_events.csv')
df_prices = pd.read_csv(DATA_PATH + 'item_prices.csv')
df_sales = pd.read_csv(DATA_PATH + 'item_sales.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
def reduce_mem_usage(df, turn_cat = False, silence = True):
    """Itera sobre todo el dataset convirtiendo cada columna en el tipo más adecuado para ahorrar memoria
    Parameters
    ----------
    df : pd.DataFrame
        Dataframe que se quiere reducir
    turn_cat : bool, optional
        Transformación de las columnas objeto o string a category, by default False
    Returns
    -------
    pd.DataFrame
        Dataframe optimizado
    """
    start_mem = df.memory_usage().sum() / 1024**2
    # print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    for col in df.columns:
        col_type = df[col].dtype

        # Saltar columnas que ya son category o datetime
        if str(col_type) in ('category', 'datetime64[ns]'):
            continue

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                # Desactivamos el casteo a float16 por los errores de precisión
                # https://github.com/pandas-dev/pandas/issues/34618
                # if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                #     df[col] = df[col].astype(np.float16)
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        if turn_cat:
            df[col] = df[col].astype('category')
    end_mem = df.memory_usage().sum() / 1024**2
    # print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    if not silence:
        print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    return df

In [7]:
df_daily  = reduce_mem_usage(df_daily, silence=False)
df_prices = reduce_mem_usage(df_prices, silence=False)
df_sales  = reduce_mem_usage(df_sales, silence=False)

Decreased by 17.5%
Decreased by 20.0%
Decreased by 78.7%


## **Exploración inicial**

### **Tabla Ventas**

In [8]:
print(f'{df_sales.shape[0]:,} filas x {df_sales.shape[1]:,} columnas')

30,490 filas x 1,920 columnas


In [9]:
display(df_sales.head())
# df_sales.describe(include= 'all')

,id,item,category,department,store,store_code,region,d_1,d_2,d_3,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [10]:
df_sales.isna().sum().sum()      # No hay nulos

np.int64(0)

In [11]:
df_sales.dtypes.head(10)

,0
id,object
item,object
category,object
department,object
store,object
store_code,object
region,object
d_1,int16
d_2,int16
d_3,int16


In [12]:
df_sales.category.value_counts()

,count
category,
SUPERMARKET,14370
HOME_&_GARDEN,10470
ACCESORIES,5650


In [13]:
df_sales.department.value_counts().sort_index()

,count
department,
ACCESORIES_1,4160
ACCESORIES_2,1490
HOME_&_GARDEN_1,5320
HOME_&_GARDEN_2,5150
SUPERMARKET_1,2160
SUPERMARKET_2,3980
SUPERMARKET_3,8230


In [14]:
df_sales.store_code.value_counts()

,count
store_code,
NYC_1,3049
NYC_2,3049
NYC_3,3049
NYC_4,3049
BOS_1,3049
BOS_2,3049
BOS_3,3049
PHI_1,3049
PHI_2,3049


In [15]:
df_sales.store.value_counts()

,count
store,
Greenwich_Village,3049
Harlem,3049
Tribeca,3049
Brooklyn,3049
South_End,3049
Roxbury,3049
Back_Bay,3049
Midtown_Village,3049
Yorktown,3049


In [16]:
df_sales.region.value_counts()

,count
region,
New York,12196
Boston,9147
Philadelphia,9147


In [17]:
df_sales.groupby('store')['item'].count()

,item
store,
Back_Bay,3049
Brooklyn,3049
Greenwich_Village,3049
Harlem,3049
Midtown_Village,3049
Queen_Village,3049
Roxbury,3049
South_End,3049
Tribeca,3049


El dataset contiene **3.049 productos** organizados en **3 categorías** y **7 departamentos**,
con ventas registradas en **10 tiendas** distribuidas en **3 regiones** (New York, Boston y Philadelphia).

El catálogo de productos es **idéntico en todas las tiendas**, aunque New York tiene 4 tiendas frente
a las 3 de Boston y Philadelphia.

### **Tabla Precios**

In [18]:
print(f'{df_prices.shape[0]:,} filas x {df_prices.shape[1]:,} columnas')

6,965,706 filas x 5 columnas


In [19]:
df_prices = df_prices.sort_values(by = 'yearweek')

In [20]:
df_prices.head()

,item,category,store_code,yearweek,sell_price
1457688,ACCESORIES_1_422,ACCESORIES,NYC_3,201105.00,3.82
1743430,SUPERMARKET_1_049,SUPERMARKET,NYC_3,201105.00,2.62
2972192,HOME_&_GARDEN_1_375,HOME_&_GARDEN,BOS_1,201105.00,9.96
1425008,ACCESORIES_1_286,ACCESORIES,NYC_3,201105.00,0.93
3280396,SUPERMARKET_3_005,SUPERMARKET,BOS_1,201105.00,2.38


In [21]:
df_prices.isna().sum()

,0
item,0
category,0
store_code,0
yearweek,243920
sell_price,0


In [22]:
df_prices.sell_price.value_counts()

,count
sell_price,
2.38,221088
3.58,217873
3.00,189541
1.20,150387
4.78,142487
...,...
18.05,1
6.44,1
10.53,1


In [23]:
df_prices.sell_price.describe()

,sell_price
count,6965706.00
mean,5.52
std,4.35
min,0.01
25%,2.62
50%,4.20
75%,7.18
max,134.15


In [24]:
df_prices.item.value_counts()

,count
item,
HOME_&_GARDEN_1_375,2870
SUPERMARKET_3_469,2870
HOME_&_GARDEN_2_394,2870
HOME_&_GARDEN_1_381,2870
SUPERMARKET_3_136,2870
...,...
HOME_&_GARDEN_1_308,652
HOME_&_GARDEN_1_159,633
HOME_&_GARDEN_1_242,610


Los 3049 productos tienen distinto número de registros de precio —> no todos
estuvieron disponibles en todas las tiendas durante todas las semanas

### **Tabla Eventos**

In [25]:
print(f'{df_daily.shape[0]:,} filas x {df_daily.shape[1]:,} columnas')

1,913 filas x 5 columnas


In [26]:
df_daily.columns

Index(['date', 'weekday', 'weekday_int', 'd', 'event'], dtype='object')

In [27]:
df_daily.isna().sum()

,0
date,0
weekday,0
weekday_int,0
d,0
event,1887


In [28]:
display(df_daily.head())
df_daily.describe(include= 'all')

,date,weekday,weekday_int,d,event
0,2011-01-29,Saturday,1,d_1,NaN
1,2011-01-30,Sunday,2,d_2,NaN
2,2011-01-31,Monday,3,d_3,NaN
3,2011-02-01,Tuesday,4,d_4,NaN
4,2011-02-02,Wednesday,5,d_5,NaN


,date,weekday,weekday_int,d,event
count,1913,1913,1913.00,1913,26
unique,1913,7,NaN,1913,5
top,2016-04-24,Saturday,NaN,d_1913,SuperBowl
freq,1,274,NaN,1,6
mean,NaN,NaN,4.00,NaN,NaN
std,NaN,NaN,2.00,NaN,NaN
min,NaN,NaN,1.00,NaN,NaN
25%,NaN,NaN,2.00,NaN,NaN
50%,NaN,NaN,4.00,NaN,NaN
75%,NaN,NaN,6.00,NaN,NaN


In [29]:
df_daily.dtypes

,0
date,object
weekday,object
weekday_int,int8
d,object
event,object


In [30]:
df_daily['date'] = pd.to_datetime( df_daily['date'])
df_daily = df_daily.sort_values(by = 'date')
# como esta ordenado luego me tiene que funcionar directamente lo de comprobar si esta completo

In [31]:
print(df_daily['date'].min())
print(df_daily['date'].max())
print(df_daily['date'].max() - df_daily['date'].min())

2011-01-29 00:00:00
2016-04-24 00:00:00
1912 days 00:00:00


El calendario tiene 1.913 días —> comprobamos que no falta ningún día.

In [32]:
def comprobar_fechas(c):
    inicio = c.min()
    fin = c.max()
    completo = pd.date_range(inicio, fin)
    print((completo == c).all())

comprobar_fechas(df_daily['date'])

True


In [33]:
df_daily['event'].value_counts(dropna= False)


,count
event,
NaN,1887
SuperBowl,6
Ramadan starts,5
Thanksgiving,5
NewYear,5
Easter,5


La gran mayoría de días no tienen ningún evento registrado — solo 26 días de los 1.913 tienen evento.

A lo mejor podríamos incorporar más información al calendario,
como por ejemplo si llovío o el tiempo, pero para más adelante cuando tengamos un df bien construido

In [34]:
display(df_daily[df_daily['event'] == 'Ramadan starts'])
display(df_daily[df_daily['event'] == 'SuperBowl'])
display(df_daily[df_daily['event'] == 'Thanksgiving'])
display(df_daily[df_daily['event'] == 'NewYear'])
display(df_daily[df_daily['event'] == 'Easter'])


,date,weekday,weekday_int,d,event
184,2011-08-01,Monday,3,d_185,Ramadan starts
538,2012-07-20,Friday,7,d_539,Ramadan starts
892,2013-07-09,Tuesday,4,d_893,Ramadan starts
1247,2014-06-29,Sunday,2,d_1248,Ramadan starts
1601,2015-06-18,Thursday,6,d_1602,Ramadan starts


,date,weekday,weekday_int,d,event
8,2011-02-06,Sunday,2,d_9,SuperBowl
372,2012-02-05,Sunday,2,d_373,SuperBowl
736,2013-02-03,Sunday,2,d_737,SuperBowl
1100,2014-02-02,Sunday,2,d_1101,SuperBowl
1464,2015-02-01,Sunday,2,d_1465,SuperBowl
1835,2016-02-07,Sunday,2,d_1836,SuperBowl


,date,weekday,weekday_int,d,event
299,2011-11-24,Thursday,6,d_300,Thanksgiving
663,2012-11-22,Thursday,6,d_664,Thanksgiving
1034,2013-11-28,Thursday,6,d_1035,Thanksgiving
1398,2014-11-27,Thursday,6,d_1399,Thanksgiving
1762,2015-11-26,Thursday,6,d_1763,Thanksgiving


,date,weekday,weekday_int,d,event
337,2012-01-01,Sunday,2,d_338,NewYear
703,2013-01-01,Tuesday,4,d_704,NewYear
1068,2014-01-01,Wednesday,5,d_1069,NewYear
1433,2015-01-01,Thursday,6,d_1434,NewYear
1798,2016-01-01,Friday,7,d_1799,NewYear


,date,weekday,weekday_int,d,event
435,2012-04-08,Sunday,2,d_436,Easter
792,2013-03-31,Sunday,2,d_793,Easter
1177,2014-04-20,Sunday,2,d_1178,Easter
1527,2015-04-05,Sunday,2,d_1528,Easter
1884,2016-03-27,Sunday,2,d_1885,Easter


Aquí lo que podemos apreciar es que tenemos una fecha por año y que no siempre coincide la fecha
en la cae el evento todos los años,excepto en el caso de Año Nuevo.

Tenemos 5 registros de cada uno menos de la superbowl porque al ser en febrero lo tenemos tanto en la parte de 2011
(a partir de febrero prácticamente) como en la parte de 2016 (hasta marzo)

Cada evento aparece una vez por año excepto SuperBowl, que al caer en febrero
aparece tanto en 2011 como en 2016 al ser los extremos del dataset.

In [35]:
(df_daily['d'].str[2:].astype('int') == pd.Series(range(1,1914))).all()
# la columna d son los numeros normales ordenados y no falta ninguno.

np.True_

In [36]:
df_daily.groupby('weekday')['weekday_int'].value_counts()

,,count
weekday,weekday_int,
Friday,7,273
Monday,3,273
Saturday,1,274
Sunday,2,274
Thursday,6,273
Tuesday,4,273
Wednesday,5,273


Vemos que es información redundante, por lo que eliminamos la columna 'weelday_int':

In [37]:
df_daily = df_daily.drop(columns=['weekday_int'])

In [38]:
def yearweek(dt):
    offsetdt = dt + timedelta(days=2)
    return offsetdt.strftime("%Y%W")

df_daily['yearweek'] = df_daily['date'].apply(lambda x: yearweek(x)).astype(float)

df_daily

,date,weekday,d,event,yearweek
0,2011-01-29,Saturday,d_1,NaN,201105.00
1,2011-01-30,Sunday,d_2,NaN,201105.00
2,2011-01-31,Monday,d_3,NaN,201105.00
3,2011-02-01,Tuesday,d_4,NaN,201105.00
4,2011-02-02,Wednesday,d_5,NaN,201105.00
...,...,...,...,...,...
1908,2016-04-20,Wednesday,d_1909,NaN,201616.00
1909,2016-04-21,Thursday,d_1910,NaN,201616.00
1910,2016-04-22,Friday,d_1911,NaN,201616.00
1911,2016-04-23,Saturday,d_1912,NaN,201617.00


#### Añadir eventos

In [39]:
us_holidays = holidays.US(years=range(2011, 2017))

# Días festivos que cubre la librería en nuestro rango de fechas
festivos_df = pd.DataFrame([{'date': fecha, 'holiday': nombre}
                            for fecha, nombre in us_holidays.items()
                            ]).sort_values('date')

In [40]:
print("Eventos que ya tenemos:")
print(df_daily['event'].value_counts(dropna=False))

print("\nEventos que añade la librería:")
print(festivos_df['holiday'].value_counts())

Eventos que ya tenemos:
event
NaN               1887
SuperBowl            6
Ramadan starts       5
Thanksgiving         5
NewYear              5
Easter               5
Name: count, dtype: int64

Eventos que añade la librería:
holiday
New Year's Day                 6
Martin Luther King Jr. Day     6
Washington's Birthday          6
Memorial Day                   6
Independence Day               6
Labor Day                      6
Columbus Day                   6
Veterans Day                   6
Thanksgiving Day               6
Christmas Day                  6
Christmas Day (observed)       2
New Year's Day (observed)      1
Veterans Day (observed)        1
Independence Day (observed)    1
Name: count, dtype: int64


Añadimos festivos oficiales de EEUU con la librería `holidays`.
La librería incluye festivos que ya teníamos (***NewYear***, ***Thanksgiving***) y versiones
***observed*** (días de sustitución cuando el festivo cae en fin de semana)



La ***máscara isna()*** protege los eventos que ya teníamos para no sobrescribirlos


In [41]:
us_holidays = holidays.US(years=range(2011, 2017))

def get_holiday(fecha):
    return us_holidays.get(fecha)

mask = df_daily['event'].isna()         # Para no sobreescribir los eventos que ya tenemos
df_daily.loc[mask, 'event'] = df_daily.loc[mask, 'date'].apply(get_holiday)

df_daily['event'].value_counts(dropna=False)

,count
event,
None,1846
SuperBowl,6
Washington's Birthday,6
Memorial Day,5
Independence Day,5
Ramadan starts,5
Labor Day,5
Columbus Day,5
Veterans Day,5


In [42]:
eventos_extra = {
    # climaticos
    '2012-10-29': 'Hurricane Sandy',
    '2012-10-30': 'Hurricane Sandy',
    '2015-01-26': 'Blizzard 2015',
    '2015-01-27': 'Blizzard 2015',
    '2016-01-23': 'Blizzard 2016',
    '2016-01-24': 'Blizzard 2016',
    # otros
    '2011-11-25': 'Black Friday', '2012-11-23': 'Black Friday',
    '2013-11-29': 'Black Friday', '2014-11-28': 'Black Friday',
    '2015-11-27': 'Black Friday', '2016-11-25': 'Black Friday',
    '2011-11-28': 'Cyber Monday', '2012-11-26': 'Cyber Monday',
    '2013-12-02': 'Cyber Monday', '2014-12-01': 'Cyber Monday',
    '2015-11-30': 'Cyber Monday', '2016-11-28': 'Cyber Monday',
    # sn valentin
    '2011-02-14': "Valentine's Day", '2012-02-14': "Valentine's Day",
    '2013-02-14': "Valentine's Day", '2014-02-14': "Valentine's Day",
    '2015-02-14': "Valentine's Day", '2016-02-14': "Valentine's Day",
}

mask_extra = df_daily['date'].dt.strftime('%Y-%m-%d').isin(eventos_extra.keys())
mask_sin_evento = df_daily['event'].isna()

df_daily.loc[mask_extra & mask_sin_evento, 'event'] = df_daily.loc[mask_extra & mask_sin_evento, 'date'].dt.strftime('%Y-%m-%d').map(eventos_extra)

print(df_daily['event'].value_counts(dropna=False))

event
None                          1824
SuperBowl                        6
Valentine's Day                  6
Washington's Birthday            6
Memorial Day                     5
Independence Day                 5
Ramadan starts                   5
Labor Day                        5
Columbus Day                     5
Veterans Day                     5
Thanksgiving                     5
Black Friday                     5
Cyber Monday                     5
Christmas Day                    5
NewYear                          5
Martin Luther King Jr. Day       5
Easter                           5
Hurricane Sandy                  2
Blizzard 2015                    2
Blizzard 2016                    2
Name: count, dtype: int64


## CRUZAMOS LOS DATASETS

Las tres tablas están separadas y necesitamos unirlas en un único `df`. Lo hacemos en tres pasos:

1. **Melt df_sales ->** transformamos las ventas de formato ancho (una columna por día) a largo
(una fila por día). Resultado: ~58M filas.

2. **Merge 1: Ventas + Calendario ->** añadimos a cada registro de venta la fecha real,
el día de la semana y el evento de ese día, usando la columna `d` como clave.

3. **Merge 2: + Precios ->** añadimos el precio de cada producto en cada tienda y semana,
usando `item + store_code + yearweek` como clave. Si no hay precio registrado esa semana,
el campo `sell_price` queda como NaN.

In [43]:
# Mantenemos el nivel de granularidad más bajo posible: una fila por producto × día × tienda.
# La predicción final es diaria, por lo que agregar a semanal supondría perder información.
# La columna yearweek solo se usa como clave de unión con la tabla de precios (que son semanales), nunca para agregar ventas.


# Paso 1:
df_sales_melt = pd.melt(df_sales, id_vars=df_sales.columns[:7], var_name='d', value_name='sales')

# Strings como categóricas para ahorrar memoria
for col in ['item', 'id', 'store_code', 'store', 'region', 'category', 'd']:
    df_sales_melt[col] = df_sales_melt[col].astype('category')

for col in ['item', 'store_code']:
    df_prices[col] = df_prices[col].astype('category')


# Paso 2:
df = df_sales_melt.merge(df_daily, on='d', how='left')

# Paso 3:
df = df.merge(df_prices.drop(columns='category'), on=['item', 'store_code', 'yearweek'], how='left')

In [44]:
def estacion(mes):
    if mes in [12, 1, 2]:
        return 'Invierno'
    elif mes in [3, 4, 5]:
        return 'Primavera'
    elif mes in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'

df['season'] = df['date'].dt.month.apply(estacion)

In [53]:
# En EEUU lo más habitual es cobrar dos veces al mes: en torno al día 1 y al día 15.
# Creamos ventanas de ±2 días alrededor de cada fecha de cobro para intentar capturar el efecto en el comportamiento de compra.
def pay_period(dia):
    if dia <= 3:
        return 'inicio_mes'
    elif 14 <= dia <= 16:
        return 'mitad_mes'
    else:
        return 'normal'

df['pay_period'] = df['date'].dt.day.apply(pay_period)

In [54]:
df

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,season,ingresos,is_holiday,pay_period
0,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_1,0,2011-01-29,Saturday,None,201105.00,12.74,Invierno,0.00,0,normal
1,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_2,0,2011-01-30,Sunday,None,201105.00,12.74,Invierno,0.00,0,normal
2,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_3,0,2011-01-31,Monday,None,201105.00,12.74,Invierno,0.00,0,normal
3,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_4,0,2011-02-01,Tuesday,None,201105.00,12.74,Invierno,0.00,0,inicio_mes
4,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_5,0,2011-02-02,Wednesday,None,201105.00,12.74,Invierno,0.00,0,inicio_mes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58327365,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1909,0,2016-04-20,Wednesday,None,201616.00,1.20,Primavera,0.00,0,normal
58327366,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1910,0,2016-04-21,Thursday,None,201616.00,1.20,Primavera,0.00,0,normal
58327367,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1911,0,2016-04-22,Friday,None,201616.00,1.20,Primavera,0.00,0,normal
58327368,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1912,0,2016-04-23,Saturday,None,201617.00,1.20,Primavera,0.00,0,normal


In [46]:
# rellenamos los nulos utilizando un forward fill por tienda y item, ya que el precio de un item en una tienda
# no suele cambiar mucho de una semana a otra.
# Usamos el precio de la semana anterior cuando no podamos usar ffill,
# rellenamos con el precio de la semana siguiente (backward fill)
df = df.sort_values(by=['item', 'store_code', 'yearweek'])
df['sell_price'] = df.groupby(['item', 'store_code'])['sell_price'].ffill().bfill()

In [47]:
# creamos una columna con el precio multiplicado por las ventas para tener la facturación
df['ingresos'] = df['sales'] * df['sell_price']

df.head()

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,season,ingresos
12196,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_1,0,2011-01-29,Saturday,None,201105.00,12.74,Invierno,0.00
42686,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_2,0,2011-01-30,Sunday,None,201105.00,12.74,Invierno,0.00
73176,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_3,0,2011-01-31,Monday,None,201105.00,12.74,Invierno,0.00
103666,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_4,0,2011-02-01,Tuesday,None,201105.00,12.74,Invierno,0.00
134156,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_5,0,2011-02-02,Wednesday,None,201105.00,12.74,Invierno,0.00


In [48]:
# marcamos un binario para ver si hay festivo o no en el día y poder hacer clustering
df['is_holiday'] = df['event'].notna().astype(int)

In [49]:
df.isna().sum()

,0
id,0
item,0
category,0
department,0
store,0
store_code,0
region,0
d,0
sales,0
date,0


In [50]:
df

,id,item,category,department,store,store_code,region,d,sales,date,weekday,event,yearweek,sell_price,season,ingresos,is_holiday
12196,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_1,0,2011-01-29,Saturday,None,201105.00,12.74,Invierno,0.00,0
42686,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_2,0,2011-01-30,Sunday,None,201105.00,12.74,Invierno,0.00,0
73176,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_3,0,2011-01-31,Monday,None,201105.00,12.74,Invierno,0.00,0
103666,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_4,0,2011-02-01,Tuesday,None,201105.00,12.74,Invierno,0.00,0
134156,ACCESORIES_1_001_BOS_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,South_End,BOS_1,Boston,d_5,0,2011-02-02,Wednesday,None,201105.00,12.74,Invierno,0.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58205409,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1909,0,2016-04-20,Wednesday,None,201616.00,1.20,Primavera,0.00,0
58235899,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1910,0,2016-04-21,Thursday,None,201616.00,1.20,Primavera,0.00,0
58266389,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1911,0,2016-04-22,Friday,None,201616.00,1.20,Primavera,0.00,0
58296879,SUPERMARKET_3_827_PHI_3,SUPERMARKET_3_827,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1912,0,2016-04-23,Saturday,None,201617.00,1.20,Primavera,0.00,0


In [51]:
df = df.reset_index(drop=True)
# df.to_feather('data_dsmarket/df_preprocessed.feather')
df.to_feather(DATA_PATH + 'df_preprocessed.feather')